building a gpt :)

In [2]:
# we are using the Tiny Shakespeare dataset, downloadable here: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print(len(text))

1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
# get character list
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_list = ''.join(chars)
print(f'numbber of unique characters: {vocab_size}, characters: {char_list}')

numbber of unique characters: 65, characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [6]:
# create a mapping from characters to integers (character level tokenizer)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("""hii there
jason""")) # spaces have index 1 while \n has index 0 
print(decode(encode("hii there"))) # note that even whitespace is tokenized

[46, 47, 47, 1, 58, 46, 43, 56, 43, 0, 48, 39, 57, 53, 52]
hii there


In [7]:
import torch 
data = torch.tensor(encode(text), dtype=torch.long)

In [8]:
data.shape

torch.Size([1115394])

In [9]:
data[:1000]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

In [10]:
# train/test split
n = int(0.9*len(data))
train_data = data[:n] # 90%
val_data = data[n:] # 10%

In [11]:
# setup context window 
block_size = 8
train_data[1:block_size+1]

tensor([47, 56, 57, 58,  1, 15, 47, 58])

In [12]:
# with a larger context window, the model takes into account all tokens (in context window) to predict the next token. After that, the window "slides" right and cycle repeats
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is {target}")

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [13]:
torch.manual_seed(1337) # deterministic
batch_size = 4 # how many batches we train in parallel
block_size = 8 # size of context window

# training on the entire dataset would take too much time, train in random batches (chunks of randomly sampled text)
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # subtracting block size ensures we don't go out of bounds 
    # note: randint takes upper bound as default and lower bound as optional
    # note 2: size must be a tuple of ints (hence batch_size,)
    x = torch.stack([data[i:i+block_size] for i in ix]) # stack literally stacks the tensors (default axis = 0, vertically)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # +1 as targets are always the next token 
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb) # these are what we input into the transformer
print('targets:')
print(yb.shape)
print(yb) # these are what we expect as output from the transformer


inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


## Testing with Bigram Model first


In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C): for every token in input (batches x context window ("time") sized tensor), returns logits which represent a score for each possible next token

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape # unpacking dimensions of logits
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) # auto reshapes into (B*T (predictions) x C (probability of all possible next tokens (scores) per prediction))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step as we want the most recent context
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution (generating the next token based off probability distribution)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [15]:
m = BigramLanguageModel(vocab_size) # create lookup table
logits, loss = m(xb, yb) # idx, targets
print(logits.shape) # for each of the 4x8 = 32 tokens, generate all 65 probabilities of the next token
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [16]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens = 10)[0].tolist())) 
# creates the start token (\n) and asks the model to predict more tokens, returns a (1,11 tensor) that we use [0] on to flatten it into (11,)
# (11,) can then be converted to a python list, which ca then be decoded back into a string


SKIcLT;AcE


### we can definitely do better than this :)

In [17]:
# create a PyTorch optimizer that handles gradient descent for us
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [18]:
batch_size = 32 # now the inputs and targets are 4 x 32 (print xb.shape)
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')
    
    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step() 

print(loss.item())

4.648136615753174


In [19]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


CxrXERo'h'q-I3sslvju
o,SV?kGyQp$YUG,NtLCMVj?qt;?uvxB$TE?wwWnZYk.iBTWT'BQm3f:Cjm:BV;a;JGGlxTbSPJUZ!!zqe!
xBP qbs$Gy'AcOmrLwwt
p$x;Seh-onQbfM?OjKbn'NwUAW -Np3fkz$FVwAUEa-wzWC -wQo-R!v -Mj?,SPiTyZ;o-opr$mOiPJEYD-CfigkzD3p3?zvS;ADz;.y?o,ivCuC'zqHxcVT cHA
rT'Fd,SBMZyOslg!NXeF$sBe,juUzLq?w-wzP-h
ERjjxlgJzPbHxf$ q,q,KCDCU fqBOQT
SVOJW:xSVwZv'DG'NSPypDhKStKzC -$hslxIVzoivnp ,ethA:NCCGoi
tN!ljjP3fwJMwNelgUzzPGJlgihJ!d?q.d
pSPYgCuCJrIFtb
jQXg
pA.P LP,SPJi
DBcuBM:CixjJ$Jzkq,OLI3KLQLMGph$O 3DfiPHnXKuHMlyjxE


## we can still do better c:

# Introducing self-attention

how self-attention works:

each token's embedding is multiplied with key, query and value matrices (which start out random and are learned during training) to get k,q,v vectors respectively 
key: what does the token have; 
value: what does the token give;
query: what is the token looking for.

attention scores are calculated by taking the dot product of each token's query and each token's key, masked (to prevent tokens from interacting with future tokens), divided by the square root of head size (length of k,q,v vectors - larger size = more expressive, more computation)**, normalized (softmax) to get attention weights then multiplied by V to get the final output - a new representation of the token enriched with context from past tokens. It's powerful as this allows each token to communicate with past tokens in one step as compared to the old method of sequentially passing information (RNN), which is much faster and cost efficient.

**As vector length increases, dot product values get insanely large which may cause softmax saturation (gradients of softmax output become negligible)
softmax([1, 2, 3])    → [0.09, 0.24, 0.67]  # nice spread
softmax([10, 20, 30]) → [0.00, 0.00, 1.00]  # one token gets ALL attention, model stops learning
dividing by sqrt(head_size) keeps variance stable and keeps numbers from exploding while retaining the structure of how attention is spread through tokens.**
 

after the loss is calculated, the kqv matrices are then updated to get more accurate attention weights via the optimizer  

In [20]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3)) 
a = a / torch.sum(a, 1, keepdim=True)
# a creates a matrix with a lower triangular structure, which is then normalized to get the average of values in each row 
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)
# this is an implmentation of future masking: future tokens don't affect existing tokens, and c are the outputs of the average values given the context window
# examplification with column 1: first row: 2/1 = 2; second row: 2+6/2 = 4; third row: 2+6+6/3 = 4.666...

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [21]:
# self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16 # more heads more computation more cost
key = nn.Linear(C, head_size, bias=False) # linear takes in C input features and applies a linear transformation to the data, returning head_size output features
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T), transpose is done for eligble matmul
# matrix multiplication is bascially just doing the dot product of every key and query simulataneously
# wei is a matrix that tells you for each batch B, what are the attention scores of each token relative to all other tokens in the batch (T,T)


tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf')) # replaces all zeroes with -inf so softmax fn converts them to 0 (1/e^inf)
wei = F.softmax(wei, dim=-1)
# high dot product, high softmax value, token has more importance

v = value(x)
out = wei @ v # weights are just the information, multiply with v (contains the actual content of each token) to get output enriched with past token representation

out.shape

torch.Size([4, 8, 16])

# Final Code

In [31]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# torch.nn.Dropout sets some fraction of input elements to 0 ("shutting off" some neurons), preventing overfitting (regularization technique)

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 100
learning_rate = 1e-2 if max_iters < 1000 else 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0 # probability of element being zeroed
temperature = 2.0 # currently clamped to [0.1, 2.0] -> [most confident, most random]
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T), normalize it to keep values in a stable range 
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), # expand
            nn.ReLU(), # learn complex non-linear relationships
            nn.Linear(4 * n_embd, n_embd), # compress
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # residual connections; withgout it, gradients need to flow back into every layer (via multiplication) which can cause vanishing gradients (if one layer's gradient is very small)
        x = x + self.sa(self.ln1(x)) 
        x = x + self.ffwd(self.ln2(x)) # now, only the change needs to be learntinstead of everything again
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            temp = max(0.1, min(temperature, 2.0)) 
            logits = logits / temp # temperature control
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5090, val loss 2.5058
step 300: train loss 2.4192, val loss 2.4332
step 400: train loss 2.3497, val loss 2.3559
step 500: train loss 2.2965, val loss 2.3128
step 600: train loss 2.2409, val loss 2.2498
step 700: train loss 2.2049, val loss 2.2187
step 800: train loss 2.1635, val loss 2.1872
step 900: train loss 2.1243, val loss 2.1500
step 1000: train loss 2.1029, val loss 2.1297
step 1100: train loss 2.0692, val loss 2.1176
step 1200: train loss 2.0380, val loss 2.0794
step 1300: train loss 2.0242, val loss 2.0636
step 1400: train loss 1.9938, val loss 2.0378
step 1500: train loss 1.9701, val loss 2.0310
step 1600: train loss 1.9634, val loss 2.0489
step 1700: train loss 1.9415, val loss 2.0141
step 1800: train loss 1.9076, val loss 1.9936
step 1900: train loss 1.9117, val loss 1.9894
step 2000: train loss 1.8851, val loss 1.9955
step 2100: train loss 1.

## note: all samples were generated with these hyperparameters
batch_size = 16; 
block_size = 32;
max_iters = 3000;
learning_rate = 1e-2 if max_iters < 1000 else 1e-3;
n_embd = 64;
n_head = 4;
n_layer = 4;
dropout = 0 

# Sample generated at temperature = 0.1:

step 2999: train loss 1.8022, val loss 1.9238

MENENIUS:
I have the strent the heart the be the streath and the his be the be streath the be the be the be the be the strown the his the be the be strent the be the be the be the be the bother the be the seed the with the be the be the be streath the be the be the bother the be the be the make the be the with the be the be the strown the here the streath and the hath the be the be the be the streath the be the be the bother the be the be the be the be the heart the be streath and the heart the heart the be the heart the be the streath strown the hath the be the be streath the here the be the be the be the see his the be the be the be the heart the with the be the streath the be streath the heart the heart the heart the here the father and the here the be the sear the dear the bother the bother the be the be to the see the heart the heart the be more the be the be the with the be streath the be the heart the bother the heart the here the heart the heart and the heart the be the be streath the heart the be streath the be the be strown the be the be the bother the be the heart the be the be the streath the be the bother art the heart the be mother the couse to the heart the be the heart the be to the be the heart the heart the heart the heart the heart the be the be the see heart the with the streath the be the be streath the be the be the with the heart the be the stard the hath the heart the heart the heart the heart the be the be the be the be streath the heart the heart the heart the be the be the be be the be the be the see the with the her the strown the here the be the be the be the be the with the couse to the heart the the bear the be the streath and the bother the his be the be the heart the be the be the strown the his the be the heart the heart the here the be the be the be the streath the be the be the be the heart the be the bother the be the be the heart the be the be streath the be the be the be the be the be the be the be the be the heart the be the










# Sample generated at temperature = 1.0:

step 2999: train loss 1.8022, val loss 1.9238

DUSCAM:
Do hen you, I was bast frolg;
I am to I pronged, you have with strebt. I'll kew this begity,

BUCKING Reamoling it o'tt drumisg;
So be the ross; nol'st, seld-begazes!
The

JUP LOYCERLANIO:
There's dest brack'd dirt! be to mecher;
Shose allowatled-brath, two spink,
Mysherd tild to as my long tabied and and in hath
'Tis on sopply-fore lought, starry,
What us thou sI with duschereng mad time ful I Vot spess they piss night.

CAMILIUS:
Kchombher, then art word! says, is we ee hand the kill coufe;
Antague at as we euch,sir that you loss to bond.

CANIO:
You and pust hespily, lave hoscieng bloudd;
And bitt boody for hear
lent the you sith if-pastlew a to to rift is blood for these bund;
Afth prokes of any Milness me, Now mostarre of that take
But End, makes, and my cad be ut:
For I you is litty is there, the our quiate,
here; hath any theren wormooth, greate fid my bewn.

ROvOMEO:
Richmeund your hear-e sessem blood, all sods, good tee to nat ply
To his armetd tay ssrans
With hy most Maresoughmand slear chall,--mostine
That bidsp to full to may the for helt
And batter of times
for to dobty stand reng timess vans?
Thed as spett take, cham thou out you; will stirranc agmmed bloods;
Yord for all as lat stoment; horgrone.
Gill wort at the desed deeds:
Thy slewick hath on learime. thereble this gentle.

GRITHORY IO:
My came and with wart; I but to your burde of is I will rep til se sapurain ward myself.
Whid not with foom thou cack:
Why spill if hereat am my bidstatle littlistired hath go.

FirsgaW:
Before; bid therealt, which the with went;
Go her wich mine wouce him hild Comen's rest you
To by bohby this bucht'd.
Whrive having not the cather; is whelp is stad my arts.
Lay like bing with, tI where stin me boke of you lords.

MENENIUS:
Nen Edward my ben it the well! I
will hees, be ve a be cersing a false heart
Agaist a take hatt arouse if yee;
The livisk my rie ex, no your they, candeds?
The my art srest you gund, every
Go me shall with is eye is tuse her,
Duke nongue 








# Sample generated at temperature = 2.0:

step 2999: train loss 1.8022, val loss 1.9238

QUSETMFuDUDIUS:
Show, ilteliquf, wlacch,. Mysu-grAct,
Tip:
O, thy,;
Dh; Britt'd-' NrBe! thi pe gity, u Faver't excol? For To-ttrujusib?
At bavAt:
Frosably:lace, seld-Cdgaze-Dilk W
UCplur despiinjcal,
xTUsudge defa-d'dzeno!:.
I to macce?; Riosau wutichside
tst. Fawfusp'nk, Mysiryant,; frour: Lnk.tn ttab-mast; taduribp theg'sMiremws;pley-ft!
Ille, wher Warc3Y,; hy kfocp.'WI
Jun'-dus weak,
Mm dugiltl uf IfVetle
Assforty; is a wouldw's dexamfacKAsdinh:
ister: houtwore! samercice I
ee I wag! Thay
Anderfertllt'gxolate a weueuch,sir kqBughfeal
so tichBElf:
What moobj'sn ypush;
Csui,miclave hoscie:I Is O;
Ein, me ftsevesh; dwir;
W
leWit' dcyver, tphir-phasldw,They,
Ayd fpers blooCEff! theselemn.

JfIhJErTOFird Pacpo, lord'sm grNf; Row ard; accuardm,
Yai3 Et,gar!
Ue sruar, my Cpd wemblood Hin y? I S&ly, blide unrry?
VINom finrath'sh.
W;ful, and sple,
Tidwnmooor ag'sT
kift wich pwgans? ve gpok?

WEMund Bifal, I -eglas ef Gno'h, alfe.
Dficg.
WhAe haven: I
Py, I'xNuaghoe:d; andssrfns
Wtell you, olMacesoughm'Evasle!
Julyes,--moddink: W cibicsp'l-ver?
TesqueMy bravEy helt
Laqf,gue? as ilives
forsemed,-
Novednclrens timlscgvansd,Tnedeat spet'st;Shalch;
Thm?F O!
Wy mad, in stirifn!
GUMmb crapeetcalloib-boias. Is;, theP'm ne;
Prosm tho vilselow lack. pude;wOhded
ginsmy I palctf! cucurnle: ime., qukkble this gen.
Awl capblewe Iik is came witt dW: ware
thinfe toGulCs burge'orp?

ICwnrOsron Til swoskpuraise?-his
Dsiwwide
by ;py d. soo,
Latzoy. Ack:
Wm my.RCI if, Glask ampecunius-atleal. thws, you hat pow.
'FH!
gaWous:
Leatro's : me ald, kmghmves Twiming:not k't.
Raxbessm a I
tucwarrEY
Y yLrocKed?
Mrife th
crocch'shb'j. And?
Brt'd.t Crivw!H,
im, notAth of that; MMy?Lel$ is stad Aswaitsw-ha;
All ibifw; therinI! bricts, . Hehy kinoms Pizim.
GWhajcEY:
I bed-nig$st Mick;,fold stifirciP!'

Gi'tble,sBick, that

HLeRD
OMICXY:
Tell mpodagexs
Nack apt, apwayo Genie$ beFffle.
Alt, you, be ei,,nkdyUlf they, cavew
sirsss-myfajansr;
Kt I which aev
zeran3 e'sy, Iilfought?
Ctbilizus: Jeks, 'but ingabm